# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
!pip install -q duckdb huggingface_hub

In [12]:
from google.colab import userdata
from huggingface_hub import list_repo_files

HF_TOKEN = userdata.get("HF_TOKEN")

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
print(len(files), "files")
for f in files[:15]:
    print(f)
print("...")
print([f for f in files if "dim_" in f or "query_90d" in f])

24 files
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
...
['dim_clients.parquet', 'dim_content.parquet', 'fact_content_query_90d.parquet']


In [13]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

# adjust the path to match cell 1's output
dim_clients = "hf://datasets/FlyRank/internship-warehouse/dim_clients*.parquet"

print(con.sql(f"SELECT COUNT(*) AS n FROM '{dim_clients}'").df())
print(con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM '{dim_clients}'
    ORDER BY gsc_data_start
    LIMIT 10
""").df())

     n
0  104
            client_hash_id gsc_data_start ga4_data_start
0  client_9958f0a7ae1df715     2025-01-27     2025-10-29
1  client_ff644d8251367cbb     2025-01-27     2025-10-29
2  client_73cda7b4e4f265ea     2025-02-11     2026-03-24
3  client_fef1a8f436438636     2025-03-11     2026-03-06
4  client_62f4a7e64f5e0096     2025-06-07            NaT
5  client_b10cb2997d0c7c86     2025-06-18     2025-11-15
6  client_65de48885f4ef01b     2025-06-21     2026-02-19
7  client_c182d11e4862a37d     2025-06-21     2026-02-20
8  client_3197e6291363b4db     2025-06-29     2025-11-09
9  client_625b6439094e23e4     2025-07-01     2026-02-19


In [14]:
dim_clients = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

# 1. see every column, including the access flags
print(con.sql(f"DESCRIBE SELECT * FROM '{dim_clients}'").df())

# 2. history depth per client, as of the last fact date
print(con.sql(f"""
    SELECT
      COUNT(*)                                                   AS n_clients,
      COUNT(gsc_data_start)                                      AS has_gsc_start,
      COUNT(ga4_data_start)                                      AS has_ga4_start,
      SUM((date_diff('month', gsc_data_start, DATE '2026-06-30') >= 6)::INT)  AS gsc_6m_plus,
      SUM((date_diff('month', gsc_data_start, DATE '2026-06-30') >= 12)::INT) AS gsc_12m_plus
    FROM '{dim_clients}'
""").df())

           column_name column_type null   key default extra
0       client_hash_id     VARCHAR  YES  None    None  None
1            is_active     BOOLEAN  YES  None    None  None
2       has_gsc_access     BOOLEAN  YES  None    None  None
3       has_ga4_access     BOOLEAN  YES  None    None  None
4       access_profile     VARCHAR  YES  None    None  None
5  client_created_date        DATE  YES  None    None  None
6  client_updated_date        DATE  YES  None    None  None
7       gsc_data_start        DATE  YES  None    None  None
8       ga4_data_start        DATE  YES  None    None  None
   n_clients  has_gsc_start  has_ga4_start  gsc_6m_plus  gsc_12m_plus
0        104             67             51         40.0           9.0


In [15]:
fact_mar = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

print(con.sql(f"""
    SELECT ga4_data_available,
           COUNT(*)                      AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM '{fact_mar}'
    GROUP BY 1 ORDER BY 1
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ga4_data_available   n_rows  n_clients
0               False  6408671         43
1                True   413966         41
2                <NA>  3018741         22


In [16]:
fact_all = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
print(con.sql(f"""
    SELECT COUNT(*) AS n_clients,
           SUM((date_diff('month', first_day, last_day) >= 6)::INT) AS six_m_plus
    FROM (SELECT client_hash_id, MIN(report_date) AS first_day, MAX(report_date) AS last_day
          FROM '{fact_all}' GROUP BY 1)
"""). df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_clients  six_m_plus
0         70        39.0


In [17]:
dim_clients = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"
fact_mar = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

# A: real overlap between GSC and GA4 start dates, and the access flags
print(con.sql(f"""
    SELECT gsc_data_start IS NOT NULL AS has_gsc_start,
           ga4_data_start IS NOT NULL AS has_ga4_start,
           has_gsc_access, has_ga4_access, COUNT(*) AS n
    FROM '{dim_clients}'
    GROUP BY ALL ORDER BY n DESC
""").df())

# B: how many different flag values does each client show in March?
print(con.sql(f"""
    SELECT n_flag_values, COUNT(*) AS n_clients
    FROM (
      SELECT client_hash_id,
             COUNT(DISTINCT COALESCE(CAST(ga4_data_available AS VARCHAR), 'NULL')) AS n_flag_values
      FROM '{fact_mar}'
      GROUP BY 1
    )
    GROUP BY 1 ORDER BY 1
""").df())

    has_gsc_start  has_ga4_start  has_gsc_access  has_ga4_access   n
0            True           True            True            True  48
1           False          False           False           False  23
2            True          False            True           False  10
3           False          False            <NA>            <NA>   6
4           False          False            True           False   4
5            True          False           False           False   3
6            True          False            <NA>            <NA>   3
7           False           True            True            True   2
8            True          False            True            True   2
9            True           True            <NA>            <NA>   1
10          False          False           False            True   1
11          False          False            True            True   1
   n_flag_values  n_clients
0              1         14
1              2         31
2              3   

In [18]:
fact_mar = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

# 1. every column, with its type
print(con.sql(f"DESCRIBE SELECT * FROM '{fact_mar}'").df())

# 2. the busiest page in March, first 15 days, one column per line
one = con.sql(f"""
    SELECT content_hash_id FROM '{fact_mar}'
    GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 1
""").fetchone()[0]
print(con.sql(f"""
    SELECT * FROM '{fact_mar}'
    WHERE content_hash_id = '{one}'
    ORDER BY report_date LIMIT 15
""").df().T)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                                0                         1   \
report_date                    2026-03-01 00:00:00       2026-03-02 00:00:00   
client_hash_id             client_73cda7b4e4f265ea   client_73cda7b4e4f265ea   
content_hash_id           content_7a105f548d9c6916  content_7a105f548d9c6916   
client_has_gsc                                True                      True   
client_has_ga4                               False                     False   
gsc_data_available                            True                      True   
ga4_data_available                            <NA>                      <NA>   
gsc_impressions                                125                       383   
gsc_clicks                                       1                         1   
gsc_sum_position                               616                      2257   
gsc_avg_position                             4.928                   5.89295   
ga4_pageviews                           

In [19]:
dim_clients = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"
fact_all = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"

print(con.sql(f"""
    SELECT COUNT(*)                         AS n_rows,
           COUNT(DISTINCT client_hash_id)   AS n_distinct_clients
    FROM (SELECT DISTINCT client_hash_id FROM '{fact_all}')
"""). df())

# LEFT JOIN anti-pattern: clients in dim_clients with NO fact rows
print(con.sql(f"""
    SELECT COUNT(*) AS clients_with_no_fact_rows
    FROM '{dim_clients}' d
    LEFT JOIN (SELECT DISTINCT client_hash_id FROM '{fact_all}') f
      ON d.client_hash_id = f.client_hash_id
    WHERE f.client_hash_id IS NULL
"""). df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_rows  n_distinct_clients
0      70                  70


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   clients_with_no_fact_rows
0                         34


In [20]:
fact_all = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
dim_clients = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

con.sql(f"CREATE TABLE fact_clients AS SELECT DISTINCT client_hash_id FROM '{fact_all}'")

print(con.sql(f"""
    SELECT d.gsc_data_start IS NOT NULL AS has_gsc_start,
           d.ga4_data_start IS NOT NULL AS has_ga4_start,
           COUNT(*) AS n_clients
    FROM fact_clients f
    JOIN '{dim_clients}' d ON d.client_hash_id = f.client_hash_id
    GROUP BY ALL ORDER BY n_clients DESC
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   has_gsc_start  has_ga4_start  n_clients
0           True           True         49
1           True          False         18
2          False           True          2
3          False          False          1


## 1. Question

*The research question and the decision it supports.*

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
